## Librerías a usar
---

In [2]:
# ============================================================
#  MLE para el Modelo de Vasicek (SHGO + Nelder-Mead + Numba)
# ============================================================

# --- Instalación de Numba (solo en Colab / Jupyter) ---
import sys
if 'google.colab' in sys.modules:
    !pip install numba -q

# --- Imports ---
import os
import time
import numpy as np
from random import random
from math import sqrt, log, exp, pi
from scipy.optimize import minimize, shgo
from numba import njit, prange

# ============================================================
# 1. DATOS
# ============================================================
if not os.path.exists('datosV.txt'):
    print("No se encontró 'datosV.txt'. Generando datos sintéticos...")
    np.random.seed(43)
    m_real     = 6.5
    sigma_real = 0.05
    c_real     = -0.5
    delta_sim  = 0.001
    n_datos    = 200

    x = m_real + 0.05
    datos = [x]
    for _ in range(n_datos - 1):
        t = 0.0
        while t < 1.0:
            x = x + c_real*(x - m_real)*delta_sim \
                  + sigma_real*np.sqrt(delta_sim)*np.random.normal()
            t += delta_sim
        datos.append(x)
    np.savetxt('datosV.txt', np.array(datos))
    print(f"Datos generados -> rango: [{min(datos):.4f}, {max(datos):.4f}], "
          f"media: {np.mean(datos):.4f}, std: {np.std(datos):.4f}")

l = np.loadtxt('datosV.txt')
print(f"Datos cargados: n={len(l)}, min={l.min():.4f}, max={l.max():.4f}, "
      f"media={l.mean():.4f}, std={l.std():.4f}")

No se encontró 'datosV.txt'. Generando datos sintéticos...
Datos generados -> rango: [6.3787, 6.6323], media: 6.4919, std: 0.0516
Datos cargados: n=200, min=6.3787, max=6.6323, media=6.4919, std=0.0516


## Funciones
---
Dividimos en funciones para poder paralelizar el código por numba.

In [3]:
# ============================================================
# 2. FUNCIONES COMPILADAS CON NUMBA
# ============================================================
@njit(cache=True, fastmath=True)
def k(z):
    return (1.0 / sqrt(2.0 * pi)) * exp(-z * z / 2.0)

@njit(cache=True, fastmath=True)
def densi(z, ll):
    n = len(ll)
    h = 1.06 * np.std(ll) / (n ** 0.2)
    if h < 1e-12:
        h = 1e-12
    suma = 0.0
    for i in range(n):
        suma += k((z - ll[i]) / h)
    return suma / (n * h)

@njit(cache=True, fastmath=True)
def f(x, c, u):
    return c * (x - u)

@njit(cache=True, fastmath=True)
def g(x, o):
    return o

@njit(parallel=True, fastmath=True, cache=True)
def F_par(x, delta, ns, l, s):
    np.random.seed(s)
    n  = len(l)
    c, u, o = x[0], x[1], x[2]
    o0 = sqrt(delta)
    NN = int(1.0 / delta)
    suma = 0.0

    all_W = np.random.normal(0.0, o0, size=(n - 1, ns, NN))

    for i in range(n - 1):
        z = l[i]
        ll  = np.empty(ns)
        ll1 = np.empty(ns)

        for nn in prange(ns):
            s_curr  = z
            s1_curr = z
            for j in range(NN):
                W = all_W[i, nn, j]
                s_curr  = s_curr  + f(s_curr,  c, u) * delta + g(s_curr,  o) * W
                s1_curr = s1_curr + f(s1_curr, c, u) * delta - g(s1_curr, o) * W
            ll[nn]  = s_curr
            ll1[nn] = s1_curr

        q  = densi(l[i + 1], ll)
        q1 = densi(l[i + 1], ll1)
        q  = 0.5 * (q + q1)
        if q < 1e-200:
            q = 1e-200
        suma += log(q)

    return -suma

@njit(cache=True, fastmath=True)
def Fe(x, delta, l):
    n = len(l)
    c, u, o = x[0], x[1], x[2]
    if c >= 0 or o <= 0:
        return 1e12
    e_decay = exp(c * delta)
    var = -(o * o) / (2.0 * c) * (1.0 - exp(2.0 * c * delta))
    if var <= 1e-16:
        return 1e12
    st = sqrt(var)
    suma = 0.0
    for i in range(n - 1):
        mu = l[i] * e_decay + u * (1.0 - e_decay)
        resid = l[i + 1] - mu
        q = (1.0 / (st * sqrt(2.0 * pi))) * exp(-(resid * resid) / (2.0 * var))
        if q < 1e-200:
            q = 1e-200
        suma += log(q)
    return -suma

## Optimización y resultados
---

In [ ]:
# ============================================================
# 3. OPTIMIZACIÓN: SHGO (global) + Nelder-Mead (réplicas en paralelo)
# ============================================================
delta     = 0.1
bbounds0  = [(-0.9, -0.1), (5.0, 8.0), (0.01, 0.09)]

print("=" * 78)
print("Fase 1: SHGO (log-verosimilitud analítica)")
print("=" * 78)

t0 = time.time()
res = shgo(Fe, args=(delta, l), bounds=bbounds0, sampling_method='sobol')
print(f"SHGO terminó en {time.time() - t0:.2f} s")

c0, u0, o0 = res.x
f0 = -res.fun
print(f"SHGO -> c0={c0:.6f}  u0={u0:.6f}  o0={o0:.6f}  -LL={f0:.4f}")

# ============================================================
# Fase 2: Réplicas de Nelder-Mead
# ============================================================
print("\n" + "=" * 78)
print(f"Fase 2: {4} réplicas Nelder-Mead (KDE + antitéticas)")
print("=" * 78)

ns = 5000
x0 = np.array([c0, u0, o0])

# Lista donde guardaremos los resultados de cada réplica
resultados = []

for i in range(4):
    s = int(100 * random())
    t_rep = time.time()

    res1 = minimize(
        F_par, x0,
        args=(delta, ns, l, s),
        method='Nelder-Mead',
        options={'fatol': 1e-3, 'xatol': 1e-3, 'maxiter': 200}
    )

    c, u, o = res1.x
    f = -res1.fun       # Log-verosimilitud (más grande = mejor)
    dt = time.time() - t_rep

    resultados.append({
        'seed': s,
        'c': c, 'u': u, 'o': o,
        'll': f,
        'tiempo': dt,
        'exito': res1.success
    })

    # Solo un aviso breve de que la réplica terminó (no todos los iter)
    print(f"  Réplica {i+1:2d} (seed={s:2d}) terminada en {dt:6.2f}s "
          f"-> -LL={-f:.4f}")

Fase 1: SHGO (log-verosimilitud analítica)


SHGO terminó en 1.19 s
SHGO -> c0=-0.900000  u0=6.488116  o0=0.090000  -LL=265.5278

Fase 2: 4 réplicas Nelder-Mead (KDE + antitéticas)
  Réplica  1 (seed=13) terminada en 116.52s -> -LL=-351.6250
  Réplica  2 (seed=12) terminada en 109.78s -> -LL=-351.5877
  Réplica  3 (seed=54) terminada en 112.69s -> -LL=-350.9869
  Réplica  4 (seed=40) terminada en 107.57s -> -LL=-351.2108


In [5]:
# ============================================================
# 4. Selección e impresión del MEJOR resultado
# ============================================================
# Ordenamos por log-verosimilitud descendente (mejor primero)
resultados_ord = sorted(resultados, key=lambda r: r['ll'], reverse=True)
mejor = resultados_ord[0]

print("\n" + "=" * 78)
print("                MEJOR RESULTADO (mayor log-verosimilitud)")
print("=" * 78)
print(f"  Seed usado:            {mejor['seed']}")
print(f"  c (velocidad rev.):    {mejor['c']:.12f}")
print(f"  u (media largo plazo): {mejor['u']:.12f}")
print(f"  o (volatilidad):       {mejor['o']:.12f}")
print(f"  Log-verosimilitud:     {mejor['ll']:.6f}")
print(f"  Tiempo de cómputo:     {mejor['tiempo']:.2f} s")

# Comparación vs SHGO analítico
print("\n  --- Error relativo vs SHGO analítico ---")
print(f"  %err c = {100*abs(mejor['c']-c0)/abs(c0):7.3f} %")
print(f"  %err u = {100*abs(mejor['u']-u0)/abs(u0):7.3f} %")
print(f"  %err o = {100*abs(mejor['o']-o0)/abs(o0):7.3f} %")

# Tabla resumen de todas las réplicas (por si quieres ver variabilidad)
print("\n" + "=" * 78)
print("  Resumen de las 4 réplicas (ordenadas por log-verosimilitud)")
print("=" * 78)
print(f"  {'#':>2} {'seed':>5} {'c':>14} {'u':>14} {'o':>14} {'-LL':>12}")
print("  " + "-" * 66)
for k, r in enumerate(resultados_ord, 1):
    print(f"  {k:>2} {r['seed']:>5} {r['c']:>14.8f} {r['u']:>14.8f} "
          f"{r['o']:>14.8f} {-r['ll']:>12.4f}")


                MEJOR RESULTADO (mayor log-verosimilitud)
  Seed usado:            13
  c (velocidad rev.):    -0.539858391064
  u (media largo plazo): 6.491848643823
  o (volatilidad):       0.050569016070
  Log-verosimilitud:     351.624950
  Tiempo de cómputo:     116.52 s

  --- Error relativo vs SHGO analítico ---
  %err c =  40.016 %
  %err u =   0.058 %
  %err o =  43.812 %

  Resumen de las 4 réplicas (ordenadas por log-verosimilitud)
   #  seed              c              u              o          -LL
  ------------------------------------------------------------------
   1    13    -0.53985839     6.49184864     0.05056902    -351.6250
   2    12    -0.54664345     6.49046287     0.05224551    -351.5877
   3    40    -0.58942271     6.49221497     0.05294993    -351.2108
   4    54    -0.55182173     6.48949543     0.05195297    -350.9869
